In [8]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
from bs4 import BeautifulSoup
import lxml
import pandas as pd
import re
from datetime import datetime
from selenium.webdriver.common.keys import Keys


### Step1: Get all the product links

In [10]:
import time
import random
from datetime import datetime
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Inputs to search
search_box_text = 'sports shoes for women'
website_link = 'https://www.flipkart.com/'

# Session start time
session_start_time = datetime.now().time()
print(f"Session Start Time: {session_start_time} ---------------------------> ")

# Starting the browser
driver = webdriver.Chrome()
driver.get(website_link)
driver.maximize_window()

print('Waiting for search input...')
search_input = WebDriverWait(driver, 120).until(
    EC.presence_of_element_located((By.CSS_SELECTOR, '[autocomplete="off"]'))
)

print('Typing in search input...')
search_input.send_keys(search_box_text)

print('Submitting search form...')
search_input.send_keys(Keys.RETURN)

print('Waiting for search results...')
WebDriverWait(driver, 120).until(
    EC.presence_of_element_located((By.CSS_SELECTOR, '[target="_blank"]'))
)

# --- FIXED PAGINATION BLOCK ---
print('Collecting pagination links...')
all_pagination_links = []
current_url = driver.current_url

# Safely split or format base pagination URL
if "&page=" in current_url:
    base_url = current_url.split("&page=")[0] + "&page="
else:
    base_url = current_url + "&page="

# Generate 25 explicit target pages
for i in range(1, 26):
    all_pagination_links.append(f"{base_url}{i}")

print('Pagination Links Count:', len(all_pagination_links))
print("All Pagination Links: ", all_pagination_links)
# ------------------------------

print("Collecting Product Detail Page Links...")
all_product_links = []

for link in all_pagination_links:
    driver.get(link)
    
    # Anti-bot human simulation delay
    time.sleep(random.uniform(2.5, 5.0))
    
    # Wait for the page to load by checking document.readyState
    WebDriverWait(driver, 120).until(lambda d: d.execute_script('return document.readyState') == 'complete')
    
    # Wait until layout cards are located
    WebDriverWait(driver, 120).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, 'div[data-id]'))
    )
    
    # Locate all individual product grid cards
    all_products = driver.find_elements(By.CSS_SELECTOR, 'div[data-id]')
    all_links = []
    
    for element in all_products:
        try:
            # Find the underlying anchor link within each product card
            link_element = element.find_element(By.CSS_SELECTOR, 'a')
            href = link_element.get_attribute('href')
            if href:
                all_links.append(href)
        except Exception:
            # Skip if a specific card doesn't have an anchor tag (like an ad slot)
            continue
            
    print(f"{link} Done ------> Found {len(all_links)} products")
    
    # --- FIXED DATA APPEND BLOCK ---
    all_product_links.extend(all_links)  # Push current page results to master container

print('All Product Detail Page Links Captured: ', len(all_product_links))

# Creating a DataFrame from the list
df_product_links = pd.DataFrame(all_product_links, columns=['product_links'])

# Remove any duplicates
df_product_links = df_product_links.drop_duplicates(subset=['product_links'])
print("Total Unique Product Detail Page Links Saved:", len(df_product_links))

# Save file output
df_product_links.to_csv('flipkart_product_links.csv', index=False)

driver.quit()  # Clean up browser session and processes completely
session_end_time = datetime.now().time()
print(f"Session End Time: {session_end_time} ---------------------------> ")


Session Start Time: 20:23:25.149640 ---------------------------> 
Waiting for search input...
Typing in search input...
Submitting search form...
Waiting for search results...
Pagination Links Count: 25
All Pagination Links:  ['https://www.flipkart.com/search?q=sports%20shoes%20for%20women&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=off&as=off&page=1', 'https://www.flipkart.com/search?q=sports%20shoes%20for%20women&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=off&as=off&page=2', 'https://www.flipkart.com/search?q=sports%20shoes%20for%20women&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=off&as=off&page=3', 'https://www.flipkart.com/search?q=sports%20shoes%20for%20women&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=off&as=off&page=4', 'https://www.flipkart.com/search?q=sports%20shoes%20for%20women&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=off&as=off&page=5', 'https://www.flipkart.com/search?q=sports%2

In [11]:
#session start time
session_start_time = datetime.now().time()
print(f"Session Start Time: {session_start_time} ---------------------------> ")


#reading the csv file which contain all product links
df_product_links = pd.read_csv("flipkart_product_links.csv")

# Remove the below line to scrap all the products. For demonstration purpose we are scraping only 10 products
df_product_links = df_product_links.head(10)

all_product_links = df_product_links['product_links'].tolist()
print("Collecting Individual Product Detail Information")

#starting the browser
driver = webdriver.Chrome()


complete_product_details = []
unavailable_products = []
successful_parsed_urls_count = 0
complete_failed_urls_count = 0
for product_page_link in all_product_links:

    try: 
        driver.get(product_page_link)
    
        # Wait for the page to load by checking document.readyState
        WebDriverWait(driver, 120).until(lambda d: d.execute_script('return document.readyState') == 'complete')
    
        WebDriverWait(driver, 120).until( EC.presence_of_element_located((By.CSS_SELECTOR, '[target="_blank"]')))
    
        #checking if product is available or not
        try:
            product_status =  driver.find_element(By.CLASS_NAME, 'Z8JjpR').text
            if product_status == 'Currently Unavailable' or product_status == 'Sold Out':
                unavailable_products.append(product_page_link)
                successful_parsed_urls_count += 1
                print(f"URL {successful_parsed_urls_count} completed --->")
        except:
            pass
    
        #brand
        brand =  driver.find_element(By.CLASS_NAME, 'mEh187').text
    
        #title       
        title = driver.find_element(By.CLASS_NAME, 'VU-ZEz').text
        title = re.sub(r'\s*\([^)]*\)', '', title)  #removing data withing parenthesis (color information)
    
        #price      
        price = driver.find_element(By.CLASS_NAME, 'Nx9bqj').text
        price = re.findall(r'\d+', price)
        price = ''.join(price)
    
        # Discount  
        try:
            discount = driver.find_element(By.CLASS_NAME, 'UkUFwK').text
            discount = re.findall(r'\d+', discount)
            discount = ''.join(discount)
            discount = int(discount) / 100
        except:
            discount = ''
    
        #for a new product, there will be no avg_rating and total_ratings    
        try:
            product_review_status = driver.find_element(By.CLASS_NAME, 'E3XX7J').text
            if product_review_status == 'Be the first to Review this product':
                avg_rating = ''
                total_ratings = ''
        except:
            avg_rating = driver.find_element(By.CLASS_NAME, 'XQDdHH').text
            total_ratings = driver.find_element(By.CLASS_NAME, 'Wphh3N').text.split(' ')[0]
            #remove the special character
            if ',' in total_ratings:
                total_ratings = int(total_ratings.replace(',', ''))
            else:
                total_ratings = int(total_ratings)
    
        successful_parsed_urls_count += 1
        print(f"URL {successful_parsed_urls_count} completed *******")
        complete_product_details.append([product_page_link, title, brand, price, discount, avg_rating, total_ratings])  
    except Exception as e:
        print(f"Failed to establish a connection for URL {product_page_link}:  {e}")
        unavailable_products.append(product_page_link)
        complete_failed_urls_count += 1
        print(f"Failed URL Count {complete_failed_urls_count}")


#create pandas dataframe 
df = pd.DataFrame(complete_product_details, columns = ['product_link', 'title', 'brand', 'price', 'discount', 'avg_rating', 'total_ratings'])
#duplicates processing
df_duplicate_products = df[df.duplicated(subset=['brand', 'price', 'discount', 'avg_rating', 'total_ratings'])]
df = df.drop_duplicates(subset=['brand', 'price', 'discount', 'avg_rating', 'total_ratings'])
#unavailable products
df_unavailable_products = pd.DataFrame(unavailable_products, columns=['link'])


#prining the stats
print("Total product pages scrapped: ", len(all_product_links))
print("Final Total Products: ", len(df))
print("Total Unavailable Products : ", len(df_unavailable_products))
print("Total Duplicate Products: ", len(df_duplicate_products))


#saving all the files
df.to_csv('flipkart_product_data.csv', index = False)
df_unavailable_products.to_csv('unavailable_products.csv', index = False)
df_duplicate_products.to_csv('duplicate_products.csv', index = False)


driver.close()
session_end_time = datetime.now().time()
print(f"Session End Time: {session_end_time} ---------------------------> ")

Session Start Time: 20:37:50.979785 ---------------------------> 
Failed to establish a connection for URL https://www.flipkart.com/campus-raye-women-s-sports-shoes-pillofoam-comfort-cushioned-memory-insole-lace-up-running-women/p/itme7a7578a5677a?pid=SHOHF5KZ5Y6D5VKU&lid=LSTSHOHF5KZ5Y6D5VKUMVPY8F&marketplace=FLIPKART&q=sports+shoes+for+women&store=osp%2Fiko%2Fd20&srno=s_1_1&otracker=search&otracker1=search&fm=organic&iid=en_WFFPzYhfyMt-z5-343bGrzHoLpjliutnRPPr7qsynrvRFKv9qo9Q9HGL3kLUq-RMpXTNmOmjmXq7A7BNtGKZBvUFjCTyOHoHZs-Z5_PS_w0%3D&ppt=None&ppn=None&ssid=9ac996469s0000001780446301163&qH=32e27b1148959538&ov_redirect=true:  Message: no such element: Unable to locate element: {"method":"css selector","selector":".mEh187"}
  (Session info: chrome=148.0.7778.179); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#nosuchelementexception
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff73f967de5+14895]
	chromedriver!GetHa

KeyboardInterrupt: 

In [13]:
import re
import time
import random
from datetime import datetime
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Session start time
session_start_time = datetime.now().time()
print(f"Session Start Time: {session_start_time} ---------------------------> ")

# Reading the CSV file
df_product_links = pd.read_csv("flipkart_product_links.csv")
df_product_links = df_product_links.head(10)  # Demonstration subset
all_product_links = df_product_links['product_links'].tolist()

print("Collecting Individual Product Detail Information...")
driver = webdriver.Chrome()

complete_product_details = []
unavailable_products = []
successful_parsed_urls_count = 0
complete_failed_urls_count = 0

for product_page_link in all_product_links:
    try:
        driver.get(product_page_link)
        time.sleep(random.uniform(2.5, 4.5))  # Humanized behavior delay
        
        # Wait for page to fully render ready state flags
        WebDriverWait(driver, 120).until(lambda d: d.execute_script('return document.readyState') == 'complete')
        
        # FIX: Wait explicitly for the core title context block to populate layout maps
        WebDriverWait(driver, 120).until(EC.presence_of_element_located((By.CSS_SELECTOR, "h1")))

        # --- Safe Availability Check ---
        is_unavailable = False
        page_text = driver.find_element(By.TAG_BODY if hasattr(By, 'TAG_BODY') else By.TAG_NAME, "body").text
        if "Currently Unavailable" in page_text or "Sold Out" in page_text:
            is_unavailable = True

        if is_unavailable:
            unavailable_products.append(product_page_link)
            successful_parsed_urls_count += 1
            print(f"URL {successful_parsed_urls_count} processed (Unavailable) --->")
            continue 

        # --- Safe Brand & Title Extraction ---
        try:
            # Flipkart typically isolates the brand label name in this specific class
            brand = driver.find_element(By.CSS_SELECTOR, "span.YgYvVb").text.strip()
        except:
            # Fallback configuration parameter if class structure shifts
            brand = "Generic/Unknown"
        
        # Capture full text cluster from target header mapping
        full_title_text = driver.find_element(By.CSS_SELECTOR, "h1").text.strip()
        
        # Cleanly strip brand prefix identifiers out of title metadata if present
        title = full_title_text.replace(brand, "").strip()
        title = re.sub(r'\s*\([^)]*\)', '', title)  # Strip out parenthetical text layers (e.g. color data)

        # --- Safe Price Extraction ---
        try:
            price_text = driver.find_element(By.CSS_SELECTOR, "div.Nx9bqj.CxhGGd").text
            price = ''.join(re.findall(r'\d+', price_text))
        except:
            price = "0"

        # --- Safe Discount Extraction ---
        try:
            discount_text = driver.find_element(By.CSS_SELECTOR, "div.UkUFwK.CcSTQR").text
            discount = int(re.findall(r'\d+', discount_text))[0] / 100
        except:
            discount = 0.0

        # --- Safe Ratings Extraction ---
        try:
            avg_rating = driver.find_element(By.CSS_SELECTOR, "div.XQDdHH").text.strip()
            ratings_string = driver.find_element(By.CSS_SELECTOR, "span.Wphh3N").text
            
            # Isolate raw digit strings from "X Ratings & Y Reviews" configuration blocks
            total_ratings = ratings_string.split()[0].replace(',', '')
            total_ratings = int(total_ratings)
        except:
            avg_rating = '0.0'
            total_ratings = 0

        successful_parsed_urls_count += 1
        print(f"URL {successful_parsed_urls_count} completed successfully *******")
        complete_product_details.append([product_page_link, title, brand, price, discount, avg_rating, total_ratings])

    except Exception as e:
        print(f"Failed to extract fields for URL {product_page_link}: {e}")
        unavailable_products.append(product_page_link)
        complete_failed_urls_count += 1
        print(f"Failed URL Count: {complete_failed_urls_count}")

# DataFrame creation and file writing operations
df = pd.DataFrame(complete_product_details, columns=['product_link', 'title', 'brand', 'price', 'discount', 'avg_rating', 'total_ratings'])
df_duplicate_products = df[df.duplicated(subset=['brand', 'price', 'discount'])]
df = df.drop_duplicates(subset=['brand', 'price', 'discount'])

df_unavailable_products = pd.DataFrame(unavailable_products, columns=['link'])

# Output stats
print("\n=== FINAL OUTPUT STATISTICS ===")
print("Total product pages scraped: ", len(all_product_links))
print("Final Total Valid Products: ", len(df))
print("Total Unavailable Products : ", len(df_unavailable_products))
print("Total Duplicate Products:    ", len(df_duplicate_products))

df.to_csv('flipkart_product_data.csv', index=False)
df_unavailable_products.to_csv('unavailable_products.csv', index=False)
df_duplicate_products.to_csv('duplicate_products.csv', index=False)

driver.quit()
session_end_time = datetime.now().time()
print(f"Session End Time: {session_end_time} ---------------------------> ")


Session Start Time: 21:11:22.445559 ---------------------------> 
URL 1 completed successfully *******
URL 2 completed successfully *******
URL 3 completed successfully *******
URL 4 completed successfully *******
URL 5 completed successfully *******
URL 6 completed successfully *******
URL 7 completed successfully *******
URL 8 completed successfully *******
URL 9 completed successfully *******
URL 10 completed successfully *******

=== FINAL OUTPUT STATISTICS ===
Total product pages scraped:  10
Final Total Valid Products:  1
Total Unavailable Products :  0
Total Duplicate Products:     9
Session End Time: 21:12:54.341255 ---------------------------> 


In [ ]:
print("\n===== ROUTER METHODS =====")
print([m for m in dir(router) if not m.startswith("_")])

print("\n===== INDEX TYPE =====")
print(type(router.index))

print("\n===== INDEX METHODS =====")
print([m for m in dir(router.index) if not m.startswith("_")])